In [5]:
# ==========================================
# NB_GOLD: SILVER -> GOLD (STAR SCHEMA PRODUCTION EDITION)
# ==========================================
from pyspark.sql.functions import col, year, month, dayofmonth, quarter, date_format, sha2, concat_ws, from_json, explode, trim, lower, when, sum, count, countDistinct, round as spark_round, first, max
from pyspark.sql.types import ArrayType, StructType, StructField, StringType, DoubleType, IntegerType
from delta.tables import DeltaTable
from datetime import datetime

print("==================================================")
print("BẮT ĐẦU MÔ HÌNH HÓA VÀ TÍNH TOÁN KPI TẦNG GOLD")
print("==================================================")

# --- BƯỚC 1: ĐỌC DỮ LIỆU TỪ TẦNG SILVER ---
silver_sales_df = spark.table("silver_sales")
silver_exchange_df = spark.table("silver_exchange_rate")
total_gold_input = silver_sales_df.count()

# Định nghĩa các Schema cấu trúc để chuẩn bị bóc tách JSON
customer_schema = StructType([
    StructField("name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True)
])

items_schema = ArrayType(StructType([
    StructField("product_id", StringType(), True),
    StructField("category", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("quantity", IntegerType(), True)
]))


# --- BƯỚC 2: TRIỂN KHAI MÔ HÌNH HÓA DỮ LIỆU STAR SCHEMA ---

# 1. Khởi tạo Bảng Chiều Thời Gian (Dim Date - Idempotent Overwrite)
dim_date = silver_sales_df.select(col("order_date")).distinct() \
    .withColumn("date_key", date_format(col("order_date"), "yyyyMMdd").cast("int")) \
    .withColumn("full_date", col("order_date")) \
    .withColumn("day", dayofmonth(col("order_date"))) \
    .withColumn("month", month(col("order_date"))) \
    .withColumn("year", year(col("order_date"))) \
    .withColumn("quarter", quarter(col("order_date")))

dim_date.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_dim_date")

# 2. Khởi tạo Bảng Chiều Khách Hàng (Dim Customers - Khử trùng tuyệt đối bằng groupBy theo Grain Khách hàng)
dim_customers = silver_sales_df \
    .withColumn("cust_json", from_json(col("customer_info"), customer_schema)) \
    .groupBy(
        col("cust_json.name").alias("customer_name"),
        col("cust_json.email").alias("customer_email")
    ) \
    .agg(
        first("cust_json.phone").alias("customer_phone"),
        max("customer_age").alias("customer_age"),
        first("location").alias("location"),
        max("loyalty_points").alias("loyalty_points")
    ) \
    .withColumn("customer_key", sha2(concat_ws("||", col("customer_name"), col("customer_email")), 256))

if spark.catalog.tableExists("gold_dim_customers"):
    print("[MERGE GOLD] Tiến hành Delta Merge thông tin khách hàng (SCD Type 1)...")
    target_dim_cust = DeltaTable.forName(spark, "gold_dim_customers")
    target_dim_cust.alias("target") \
        .merge(dim_customers.alias("source"), "target.customer_key = source.customer_key") \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
else:
    dim_customers.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_dim_customers")

# 3. Khởi tạo Bảng Chiều Sản Phẩm (Dim Products - Khử trùng tuyệt đối bằng groupBy theo Grain Sản phẩm)
dim_products = silver_sales_df \
    .withColumn("parsed_items", from_json(col("items"), items_schema)) \
    .withColumn("item", explode(col("parsed_items"))) \
    .groupBy(col("item.product_id").alias("product_id")) \
    .agg(first("item.category").alias("category")) \
    .withColumn("product_key", sha2(col("product_id"), 256))

if spark.catalog.tableExists("gold_dim_products"):
    print("[MERGE GOLD] Tiến hành Delta Merge danh mục sản phẩm...")
    target_dim_prod = DeltaTable.forName(spark, "gold_dim_products")
    target_dim_prod.alias("target") \
        .merge(dim_products.alias("source"), "target.product_key = source.product_key") \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
else:
    dim_products.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_dim_products")

# 4. Khởi tạo Bảng Sự Kiện Trung Tâm (Fact Sales - Gom nhóm chuẩn hóa mức độ chi tiết tránh trùng cặp Order + Product)
fact_sales_raw = silver_sales_df \
    .withColumn("cust_json", from_json(col("customer_info"), customer_schema)) \
    .withColumn("parsed_items", from_json(col("items"), items_schema)) \
    .withColumn("item", explode(col("parsed_items"))) \
    .withColumn("date_key", date_format(col("order_date"), "yyyyMMdd").cast("int")) \
    .withColumn("customer_key", sha2(concat_ws("||", col("cust_json.name"), col("cust_json.email")), 256)) \
    .withColumn("product_key", sha2(col("item.product_id"), 256))

fact_sales = fact_sales_raw.groupBy("order_id", "product_key") \
    .agg(
        first("date_key").alias("date_key"),
        first("customer_key").alias("customer_key"),
        first("device_type").alias("device_type"),
        first("payment_method").alias("payment_method"),
        first("currency").alias("currency"),
        first("discount_code").alias("discount_code"),
        first("shipping_cost").alias("shipping_cost"),
        first("order_status").alias("order_status"),
        first("feedback_score").alias("feedback_score"),
        sum("item.quantity").alias("quantity"),
        first("item.price").alias("price_usd"),
        sum(col("item.price") * col("item.quantity")).alias("item_subtotal_usd")
    )

if spark.catalog.tableExists("gold_fact_sales"):
    print("[MERGE GOLD] Tiến hành Delta Merge cập nhật dữ liệu giao dịch vào Fact Table...")
    target_fact = DeltaTable.forName(spark, "gold_fact_sales")
    target_fact.alias("target") \
        .merge(fact_sales.alias("source"), "target.order_id = source.order_id AND target.product_key = source.product_key") \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
else:
    fact_sales.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_fact_sales")

# NÂNG CẤP HIỆU NĂNG: Gom cụm sắp xếp đa chiều vật lý tối ưu hóa tốc độ Join/Filter cho BI
print("[OPTIMIZE] Đang tái cấu trúc sắp xếp dữ liệu (Z-Order Clustering) cho bảng Fact...")
spark.sql("OPTIMIZE gold_fact_sales ZORDER BY (date_key, product_key)")

print("[LOG] Đã hoàn thành tổ chức dữ liệu cấu trúc mô hình Star Schema.")


# --- BƯỚC 3: TRUY VẤN TÍNH TOÁN CHỈ SỐ BI (ÁP DỤNG BỘ LỌC LOẠI BỎ DATA ẢO) ---

f_sales = spark.table("gold_fact_sales")
d_products = spark.table("gold_dim_products")
d_date = spark.table("gold_dim_date")
d_customers = spark.table("gold_dim_customers")
s_exchange = spark.table("silver_exchange_rate")

# Bộ lọc loại bỏ dữ liệu ảo: Không lấy Failed, feedback_score chỉ từ 1-5 (hoặc null do chưa đánh giá)
fact_filtered = f_sales.filter(
    (lower(trim(col("order_status"))) != "failed") & 
    ((col("feedback_score").isNull()) | (col("feedback_score").between(1, 5)))
)

# KPI 1: Báo cáo Doanh thu VNĐ hàng tháng CHI TIẾT THEO CATEGORY
sales_enriched = fact_filtered \
    .join(d_date, "date_key") \
    .join(d_products, "product_key") \
    .join(s_exchange, ["year", "month"], "left")

gold_monthly_revenue = sales_enriched \
    .withColumn("revenue_vnd", col("item_subtotal_usd") * col("exchange_rate")) \
    .groupBy("year", "month", "category") \
    .agg(spark_round(sum("revenue_vnd"), 0).alias("Total_Revenue_VND")) \
    .orderBy("year", "month", "category")

# KPI 2: Phân tích tỷ lệ sử dụng mã khuyến mãi theo khu vực địa lý (Tính trên số lượng Order ID duy nhất)
sales_with_cust = fact_filtered.join(d_customers, "customer_key")

gold_promotion_analysis = sales_with_cust.groupBy("location").agg(
    countDistinct("order_id").alias("total_orders"),
    countDistinct(when(col("discount_code").isNotNull() & (trim(col("discount_code")) != ""), col("order_id"))).alias("promo_orders")
).withColumn(
    "promo_percent",
    spark_round(col("promo_orders") * 100 / col("total_orders"), 2)
).orderBy(col("promo_percent").desc())


# --- BƯỚC 4: GHI RA CÁC BẢNG TỔNG HỢP (ÁP DỤNG MERGE ĐỂ ĐẠT TÍNH CHẤT IDEMPOTENT) ---

if spark.catalog.tableExists("gold_monthly_revenue"):
    print("[MERGE GOLD] Tiến hành Delta Merge cập nhật bảng báo cáo doanh thu...")
    target_revenue = DeltaTable.forName(spark, "gold_monthly_revenue")
    target_revenue.alias("target") \
        .merge(
            gold_monthly_revenue.alias("source"), 
            "target.year = source.year AND target.month = source.month AND target.category = source.category"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
else:
    gold_monthly_revenue.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_monthly_revenue")

# Bảng phân tích phân bổ địa lý ghi đè (Overwrite) vì dung lượng nhỏ
gold_promotion_analysis.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_promotion_analysis")


# --- BƯỚC 5: METADATA AUDIT LOGGING ---
notebook_name = "NB_GOLD"
status = "SUCCESS"
execution_time = datetime.now()

log_data = [(notebook_name, status, execution_time, int(total_gold_input), 0)]
log_schema = ["notebook_name", "status", "execution_at", "processed_records", "quarantine_records"]

pipeline_log_df = spark.createDataFrame(log_data, schema=log_schema)
pipeline_log_df.write.format("delta").mode("append").saveAsTable("sys_pipeline_audit_logs")

print("==================================================")
print("=== TOÀN BỘ PIPELINE ĐÃ HOÀN THÀNH  ===")
print("==================================================")

StatementMeta(, 18ef6b88-ab26-4d25-b35c-dbb28a6fb352, 7, Finished, Available, Finished, False)

BẮT ĐẦU MÔ HÌNH HÓA VÀ TÍNH TOÁN KPI TẦNG GOLD
[MERGE GOLD] Tiến hành Delta Merge thông tin khách hàng (SCD Type 1)...
[MERGE GOLD] Tiến hành Delta Merge danh mục sản phẩm...
[MERGE GOLD] Tiến hành Delta Merge cập nhật dữ liệu giao dịch vào Fact Table...
[OPTIMIZE] Đang tái cấu trúc sắp xếp dữ liệu (Z-Order Clustering) cho bảng Fact...
[LOG] Đã hoàn thành tổ chức dữ liệu cấu trúc mô hình Star Schema.
[MERGE GOLD] Tiến hành Delta Merge cập nhật bảng báo cáo doanh thu...
=== TOÀN BỘ PIPELINE ĐÃ HOÀN THÀNH  ===
